# Fleet Allocation Optimization

## Public Transport Demand Planning — Kharagpur → Kolkata Corridor

This notebook converts predicted passenger demand into an operational
fleet allocation plan using mathematical optimization.

### Decision Objective

Determine the number of bus and rail services required on each corridor
segment while minimizing:

- Operating cost
- Unmet passenger demand

subject to fleet and service-capacity constraints.

In [1]:
import pandas as pd
import numpy as np
import pulp

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================
# 3. LOAD DEMAND FORECAST
# ============================================

forecast_output = pd.read_csv(
    "../outputs/demand_forecast.csv"
)

forecast_output["date"] = pd.to_datetime(
    forecast_output["date"]
)

print("Forecast data loaded successfully.")
print("Shape:", forecast_output.shape)
print("\nDate range:")
print(
    forecast_output["date"].min(),
    "to",
    forecast_output["date"].max()
)

print("\nColumns:")
print(forecast_output.columns.tolist())

forecast_output.head()

Forecast data loaded successfully.
Shape: (128, 5)

Date range:
2025-10-14 00:00:00 to 2025-10-29 00:00:00

Columns:
['date', 'route', 'passengers', 'predicted_demand', 'forecast_error']


,date,route,passengers,predicted_demand,forecast_error
0,2025-10-14,Howrah_Kolkata_Bus,6794,6227,567
1,2025-10-15,Howrah_Kolkata_Bus,5529,6683,-1154
2,2025-10-16,Howrah_Kolkata_Bus,6601,6513,88
3,2025-10-17,Howrah_Kolkata_Bus,7549,6657,892
4,2025-10-18,Howrah_Kolkata_Bus,6539,5557,982


In [3]:
# ============================================
# 4. OPERATING ASSUMPTIONS
# ============================================

BUS_CAPACITY = 50
RAIL_CAPACITY = 1000

BUS_COST = 1
RAIL_COST = 8

MAX_BUS_SERVICES = 400
MAX_RAIL_SERVICES = 80

UNMET_DEMAND_PENALTY = 20

print("Operating assumptions loaded.")
print(f"Bus capacity       : {BUS_CAPACITY}")
print(f"Rail capacity      : {RAIL_CAPACITY}")
print(f"Bus cost/service   : {BUS_COST}")
print(f"Rail cost/service  : {RAIL_COST}")
print(f"Max bus services   : {MAX_BUS_SERVICES}")
print(f"Max rail services  : {MAX_RAIL_SERVICES}")
print(f"Unmet penalty      : {UNMET_DEMAND_PENALTY}")

Operating assumptions loaded.
Bus capacity       : 50
Rail capacity      : 1000
Bus cost/service   : 1
Rail cost/service  : 8
Max bus services   : 400
Max rail services  : 80
Unmet penalty      : 20


In [4]:
# ============================================
# 5. FLEET AVAILABILITY
# ============================================

TOTAL_BUS_FLEET = 40
TOTAL_RAIL_FLEET = 50

print("Fleet availability:")
print(f"Total buses available per day : {TOTAL_BUS_FLEET}")
print(f"Total rail services available : {TOTAL_RAIL_FLEET}")

Fleet availability:
Total buses available per day : 40
Total rail services available : 50


In [5]:
# ============================================
# 6. SELECT PLANNING DAY
# ============================================

planning_date = forecast_output["date"].min()

daily_forecast = forecast_output[
    forecast_output["date"] == planning_date
].copy()

print("Planning date:", planning_date.date())
print("\nForecast demand:")
print(
    daily_forecast[
        ["route", "predicted_demand"]
    ].to_string(index=False)
)

print(
    "\nTotal forecast demand:",
    round(daily_forecast["predicted_demand"].sum())
)

Planning date: 2025-10-14

Forecast demand:
                   route  predicted_demand
      Howrah_Kolkata_Bus              6227
     Howrah_Kolkata_Rail             14524
 Kharagpur_Midnapore_Bus              8462
Kharagpur_Midnapore_Rail             11205
  Midnapore_Uluberia_Bus              6568
 Midnapore_Uluberia_Rail             14519
     Uluberia_Howrah_Bus              6623
    Uluberia_Howrah_Rail             12661

Total forecast demand: 80789


In [6]:
# ============================================
# 7. FLEET-CONSTRAINED OPTIMIZATION MODEL
# ============================================

model = pulp.LpProblem(
    "Fleet_Allocation_Optimization",
    pulp.LpMinimize
)

routes = daily_forecast["route"].tolist()

# Decision variables
bus_services = {
    route: pulp.LpVariable(
        f"bus_{route}",
        lowBound=0,
        cat="Integer"
    )
    for route in routes
}

rail_services = {
    route: pulp.LpVariable(
        f"rail_{route}",
        lowBound=0,
        cat="Integer"
    )
    for route in routes
}

unmet_demand = {
    route: pulp.LpVariable(
        f"unmet_{route}",
        lowBound=0,
        cat="Continuous"
    )
    for route in routes
}

# --------------------------------------------
# Objective
# --------------------------------------------

model += pulp.lpSum(
    BUS_COST * bus_services[r]
    + RAIL_COST * rail_services[r]
    + UNMET_DEMAND_PENALTY * unmet_demand[r]
    for r in routes
)

# --------------------------------------------
# Route-level demand constraints
# --------------------------------------------

for _, row in daily_forecast.iterrows():

    route = row["route"]
    demand = row["predicted_demand"]

    model += (
        BUS_CAPACITY * bus_services[route]
        + RAIL_CAPACITY * rail_services[route]
        + unmet_demand[route]
        >= demand
    )

# --------------------------------------------
# Fleet-wide constraints
# --------------------------------------------

model += pulp.lpSum(
    bus_services[r] for r in routes
) <= TOTAL_BUS_FLEET

model += pulp.lpSum(
    rail_services[r] for r in routes
) <= TOTAL_RAIL_FLEET

# --------------------------------------------
# Solve
# --------------------------------------------

model.solve(
    pulp.PULP_CBC_CMD(msg=False)
)

print("Optimization status:")
print(pulp.LpStatus[model.status])

Optimization status:
Optimal


In [9]:
# ============================================
# 8. EXTRACT OPTIMAL FLEET ALLOCATION
# ============================================

optimization_results = []

for route in routes:

    demand = daily_forecast.loc[
        daily_forecast["route"] == route,
        "predicted_demand"
    ].iloc[0]

    buses = int(round(pulp.value(bus_services[route])))
    rail = int(round(pulp.value(rail_services[route])))
    unmet = pulp.value(unmet_demand[route])

    capacity = (
        BUS_CAPACITY * buses
        + RAIL_CAPACITY * rail
    )

    optimization_results.append({
        "route": route,
        "forecast_demand": round(demand),
        "bus_services": buses,
        "rail_services": rail,
        "capacity_provided": capacity,
        "unmet_demand": round(unmet, 2),
        "capacity_coverage_pct": round(
    min(demand, capacity) / demand * 100,
    2
) if demand > 0 else 0
    })

optimization_results = pd.DataFrame(
    optimization_results
)

optimization_results

,route,forecast_demand,bus_services,rail_services,capacity_provided,unmet_demand,capacity_coverage_pct
0,Howrah_Kolkata_Bus,6227,4,6,6200,27.0,99.57
1,Howrah_Kolkata_Rail,14524,2,0,100,14424.0,0.69
2,Kharagpur_Midnapore_Bus,8462,0,0,0,8462.0,0.00
3,Kharagpur_Midnapore_Rail,11205,0,6,6000,5205.0,53.55
4,Midnapore_Uluberia_Bus,6568,11,6,6550,18.0,99.73
5,Midnapore_Uluberia_Rail,14519,10,14,14500,19.0,99.87
6,Uluberia_Howrah_Bus,6623,0,6,6000,623.0,90.59
7,Uluberia_Howrah_Rail,12661,13,12,12650,11.0,99.91


In [10]:
# ============================================
# NETWORK CAPACITY GAP
# ============================================

total_forecast_demand = (
    optimization_results["forecast_demand"].sum()
)

total_capacity = (
    optimization_results["capacity_provided"].sum()
)

total_unmet = (
    optimization_results["unmet_demand"].sum()
)

coverage = (
    (total_forecast_demand - total_unmet)
    / total_forecast_demand
    * 100
)

print("NETWORK CAPACITY ANALYSIS")
print("-----------------------------------")
print(f"Forecast demand       : {total_forecast_demand:,}")
print(f"Capacity provided     : {total_capacity:,}")
print(f"Unmet demand          : {total_unmet:,.0f}")
print(f"Demand coverage       : {coverage:.2f}%")
print(
    f"Capacity gap          : "
    f"{total_forecast_demand - total_capacity:,}"
)

NETWORK CAPACITY ANALYSIS
-----------------------------------
Forecast demand       : 80,789
Capacity provided     : 52,000
Unmet demand          : 28,789
Demand coverage       : 64.37%
Capacity gap          : 28,789


In [8]:
# ============================================
# 9. NETWORK-LEVEL SUMMARY
# ============================================

total_buses_used = optimization_results["bus_services"].sum()
total_rail_used = optimization_results["rail_services"].sum()
total_unmet = optimization_results["unmet_demand"].sum()

total_cost = (
    total_buses_used * BUS_COST
    + total_rail_used * RAIL_COST
    + total_unmet * UNMET_DEMAND_PENALTY
)

print("NETWORK OPTIMIZATION SUMMARY")
print("-----------------------------------")
print(f"Buses used        : {total_buses_used}")
print(f"Buses available   : {TOTAL_BUS_FLEET}")
print(f"Rail services used: {total_rail_used}")
print(f"Rail available    : {TOTAL_RAIL_FLEET}")
print(f"Total unmet demand: {total_unmet:.2f}")
print(f"Total operating cost: {total_cost:.2f}")

NETWORK OPTIMIZATION SUMMARY
-----------------------------------
Buses used        : 40
Buses available   : 40
Rail services used: 50
Rail available    : 50
Total unmet demand: 28789.00
Total operating cost: 576220.00


## 10. Corrected Multimodal Fleet Allocation Model

The initial optimization formulation treated each route-mode combination
as an independent route. This could allow a vehicle type to be assigned
to a route labelled for another mode.

To represent the transportation system more realistically, the corrected
model separates:

- Physical corridor segment
- Rail demand
- Bus demand
- Rail fleet allocation
- Bus fleet allocation

The optimization therefore assigns buses only to bus demand and rail
services only to rail demand, while both modes compete for their respective
limited fleet resources.

In [11]:
# ============================================
# 10. CREATE PHYSICAL CORRIDOR SEGMENTS
# ============================================

# Extract origin and destination from route
forecast_output[["origin", "destination", "mode"]] = (
    forecast_output["route"].str.split("_", expand=True)
)

forecast_output["segment"] = (
    forecast_output["origin"]
    + " → "
    + forecast_output["destination"]
)

# Select the same planning day
daily_forecast = forecast_output[
    forecast_output["date"] == planning_date
].copy()

print("Physical corridor segments:")
print(
    daily_forecast["segment"]
    .unique()
)

Physical corridor segments:
['Howrah → Kolkata' 'Kharagpur → Midnapore' 'Midnapore → Uluberia'
 'Uluberia → Howrah']


In [12]:
# ============================================
# 11. MODE-SPECIFIC DEMAND
# ============================================

segment_demand = (
    daily_forecast
    .pivot_table(
        index="segment",
        columns="mode",
        values="predicted_demand",
        aggfunc="sum"
    )
    .fillna(0)
    .reset_index()
)

# Make sure both columns exist
if "Bus" not in segment_demand.columns:
    segment_demand["Bus"] = 0

if "Rail" not in segment_demand.columns:
    segment_demand["Rail"] = 0

segment_demand = segment_demand[
    ["segment", "Bus", "Rail"]
]

segment_demand.columns = [
    "segment",
    "bus_demand",
    "rail_demand"
]

segment_demand

,segment,bus_demand,rail_demand
0,Howrah → Kolkata,6227,14524
1,Kharagpur → Midnapore,8462,11205
2,Midnapore → Uluberia,6568,14519
3,Uluberia → Howrah,6623,12661


In [13]:
# ============================================
# 12. CORRECTED FLEET OPTIMIZATION
# ============================================

corrected_model = pulp.LpProblem(
    "Corrected_Multimodal_Fleet_Allocation",
    pulp.LpMinimize
)

segments = segment_demand["segment"].tolist()

# --------------------------------------------
# Decision variables
# --------------------------------------------

bus_services_corrected = {
    segment: pulp.LpVariable(
        f"bus_{segment}",
        lowBound=0,
        cat="Integer"
    )
    for segment in segments
}

rail_services_corrected = {
    segment: pulp.LpVariable(
        f"rail_{segment}",
        lowBound=0,
        cat="Integer"
    )
    for segment in segments
}

unmet_bus = {
    segment: pulp.LpVariable(
        f"unmet_bus_{segment}",
        lowBound=0,
        cat="Continuous"
    )
    for segment in segments
}

unmet_rail = {
    segment: pulp.LpVariable(
        f"unmet_rail_{segment}",
        lowBound=0,
        cat="Continuous"
    )
    for segment in segments
}

In [14]:
# ============================================
# OBJECTIVE FUNCTION
# ============================================

corrected_model += pulp.lpSum(
    BUS_COST * bus_services_corrected[s]
    + RAIL_COST * rail_services_corrected[s]
    + UNMET_DEMAND_PENALTY * (
        unmet_bus[s] + unmet_rail[s]
    )
    for s in segments
)

In [15]:
# ============================================
# MODE-SPECIFIC DEMAND CONSTRAINTS
# ============================================

for _, row in segment_demand.iterrows():

    segment = row["segment"]

    bus_demand = row["bus_demand"]
    rail_demand = row["rail_demand"]

    # Bus demand can only be served by buses
    corrected_model += (
        BUS_CAPACITY * bus_services_corrected[segment]
        + unmet_bus[segment]
        >= bus_demand
    )

    # Rail demand can only be served by rail services
    corrected_model += (
        RAIL_CAPACITY * rail_services_corrected[segment]
        + unmet_rail[segment]
        >= rail_demand
    )

In [16]:
# ============================================
# FLEET-WIDE RESOURCE CONSTRAINTS
# ============================================

corrected_model += pulp.lpSum(
    bus_services_corrected[s]
    for s in segments
) <= TOTAL_BUS_FLEET

corrected_model += pulp.lpSum(
    rail_services_corrected[s]
    for s in segments
) <= TOTAL_RAIL_FLEET

In [17]:
# ============================================
# SOLVE CORRECTED MODEL
# ============================================

corrected_model.solve(
    pulp.PULP_CBC_CMD(msg=False)
)

print(
    "Corrected optimization status:",
    pulp.LpStatus[corrected_model.status]
)

Corrected optimization status: Optimal


In [18]:
# ============================================
# 13. CORRECTED OPTIMIZATION RESULTS
# ============================================

corrected_results = []

for _, row in segment_demand.iterrows():

    segment = row["segment"]

    bus_demand = row["bus_demand"]
    rail_demand = row["rail_demand"]

    buses = int(
        round(
            pulp.value(
                bus_services_corrected[segment]
            )
        )
    )

    rail = int(
        round(
            pulp.value(
                rail_services_corrected[segment]
            )
        )
    )

    bus_unmet = pulp.value(
        unmet_bus[segment]
    )

    rail_unmet = pulp.value(
        unmet_rail[segment]
    )

    bus_capacity = buses * BUS_CAPACITY
    rail_capacity = rail * RAIL_CAPACITY

    corrected_results.append({
        "segment": segment,
        "bus_demand": round(bus_demand),
        "rail_demand": round(rail_demand),
        "bus_services": buses,
        "rail_services": rail,
        "bus_capacity": bus_capacity,
        "rail_capacity": rail_capacity,
        "bus_unmet": round(bus_unmet, 2),
        "rail_unmet": round(rail_unmet, 2)
    })

corrected_results = pd.DataFrame(
    corrected_results
)

corrected_results

,segment,bus_demand,rail_demand,bus_services,rail_services,bus_capacity,rail_capacity,bus_unmet,rail_unmet
0,Howrah → Kolkata,6227,14524,0,14,0,14000,6227.0,524.0
1,Kharagpur → Midnapore,8462,11205,40,11,2000,11000,6462.0,205.0
2,Midnapore → Uluberia,6568,14519,0,14,0,14000,6568.0,519.0
3,Uluberia → Howrah,6623,12661,0,11,0,11000,6623.0,1661.0


In [19]:
# ============================================
# 14. CORRECTED NETWORK SUMMARY
# ============================================

total_bus_demand = corrected_results["bus_demand"].sum()
total_rail_demand = corrected_results["rail_demand"].sum()

total_bus_capacity = corrected_results["bus_capacity"].sum()
total_rail_capacity = corrected_results["rail_capacity"].sum()

total_bus_unmet = corrected_results["bus_unmet"].sum()
total_rail_unmet = corrected_results["rail_unmet"].sum()

total_demand = total_bus_demand + total_rail_demand
total_capacity = total_bus_capacity + total_rail_capacity
total_unmet = total_bus_unmet + total_rail_unmet

demand_coverage = (
    (total_demand - total_unmet)
    / total_demand
    * 100
)

print("CORRECTED NETWORK SUMMARY")
print("-----------------------------------")
print(f"Total bus demand       : {total_bus_demand:,.0f}")
print(f"Total rail demand      : {total_rail_demand:,.0f}")
print(f"Total forecast demand  : {total_demand:,.0f}")
print()
print(f"Bus capacity provided  : {total_bus_capacity:,.0f}")
print(f"Rail capacity provided : {total_rail_capacity:,.0f}")
print(f"Total capacity         : {total_capacity:,.0f}")
print()
print(f"Bus unmet demand       : {total_bus_unmet:,.0f}")
print(f"Rail unmet demand      : {total_rail_unmet:,.0f}")
print(f"Total unmet demand     : {total_unmet:,.0f}")
print()
print(f"Demand coverage        : {demand_coverage:.2f}%")
print()
print(f"Bus services used      : {corrected_results['bus_services'].sum()}")
print(f"Rail services used     : {corrected_results['rail_services'].sum()}")

CORRECTED NETWORK SUMMARY
-----------------------------------
Total bus demand       : 27,880
Total rail demand      : 52,909
Total forecast demand  : 80,789

Bus capacity provided  : 2,000
Rail capacity provided : 50,000
Total capacity         : 52,000

Bus unmet demand       : 25,880
Rail unmet demand      : 2,909
Total unmet demand     : 28,789

Demand coverage        : 64.37%

Bus services used      : 40
Rail services used     : 50


In [20]:
# ============================================
# 15. REUSABLE FLEET OPTIMIZATION FUNCTION
# ============================================

def optimize_fleet(
    demand_data,
    total_buses,
    total_rail
):
    """
    Optimize multimodal fleet allocation for one planning day.

    Bus demand can only be served by buses.
    Rail demand can only be served by rail services.
    """

    model = pulp.LpProblem(
        "Fleet_Allocation",
        pulp.LpMinimize
    )

    segments = demand_data["segment"].tolist()

    # Decision variables
    bus_services = {
        s: pulp.LpVariable(
            f"bus_{s}",
            lowBound=0,
            cat="Integer"
        )
        for s in segments
    }

    rail_services = {
        s: pulp.LpVariable(
            f"rail_{s}",
            lowBound=0,
            cat="Integer"
        )
        for s in segments
    }

    unmet_bus = {
        s: pulp.LpVariable(
            f"unmet_bus_{s}",
            lowBound=0
        )
        for s in segments
    }

    unmet_rail = {
        s: pulp.LpVariable(
            f"unmet_rail_{s}",
            lowBound=0
        )
        for s in segments
    }

    # Objective
    model += pulp.lpSum(
        BUS_COST * bus_services[s]
        + RAIL_COST * rail_services[s]
        + UNMET_DEMAND_PENALTY *
        (unmet_bus[s] + unmet_rail[s])
        for s in segments
    )

    # Demand constraints
    for _, row in demand_data.iterrows():

        s = row["segment"]

        model += (
            BUS_CAPACITY * bus_services[s]
            + unmet_bus[s]
            >= row["bus_demand"]
        )

        model += (
            RAIL_CAPACITY * rail_services[s]
            + unmet_rail[s]
            >= row["rail_demand"]
        )

    # Fleet constraints
    model += pulp.lpSum(
        bus_services[s]
        for s in segments
    ) <= total_buses

    model += pulp.lpSum(
        rail_services[s]
        for s in segments
    ) <= total_rail

    # Solve
    model.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    # Results
    total_unmet = sum(
        pulp.value(unmet_bus[s])
        + pulp.value(unmet_rail[s])
        for s in segments
    )

    total_bus_used = sum(
        pulp.value(bus_services[s])
        for s in segments
    )

    total_rail_used = sum(
        pulp.value(rail_services[s])
        for s in segments
    )

    total_capacity = (
        total_bus_used * BUS_CAPACITY
        + total_rail_used * RAIL_CAPACITY
    )

    total_demand = (
        demand_data["bus_demand"].sum()
        + demand_data["rail_demand"].sum()
    )

    coverage = (
        (total_demand - total_unmet)
        / total_demand
        * 100
    )

    operating_cost = (
        total_bus_used * BUS_COST
        + total_rail_used * RAIL_COST
    )

    return {
        "buses_available": total_buses,
        "rail_available": total_rail,
        "buses_used": round(total_bus_used),
        "rail_used": round(total_rail_used),
        "capacity": round(total_capacity),
        "unmet_demand": round(total_unmet),
        "coverage_pct": round(coverage, 2),
        "operating_cost": round(operating_cost, 2)
    }

In [21]:
# ============================================
# 16. BASELINE SCENARIO
# ============================================

baseline = optimize_fleet(
    segment_demand,
    total_buses=40,
    total_rail=50
)

baseline

{'buses_available': 40,
 'rail_available': 50,
 'buses_used': 40,
 'rail_used': 50,
 'capacity': 52000,
 'unmet_demand': 28789,
 'coverage_pct': np.float64(64.37),
 'operating_cost': 440.0}

In [22]:
# ============================================
# 17. FLEET EXPANSION SCENARIOS
# ============================================

scenarios = [
    {
        "scenario": "Current Fleet",
        "buses": 40,
        "rail": 50
    },
    {
        "scenario": "+10 Buses",
        "buses": 50,
        "rail": 50
    },
    {
        "scenario": "+10 Rail",
        "buses": 40,
        "rail": 60
    },
    {
        "scenario": "+20 Buses",
        "buses": 60,
        "rail": 50
    },
    {
        "scenario": "+20 Rail",
        "buses": 40,
        "rail": 70
    },
    {
        "scenario": "+10 Bus +10 Rail",
        "buses": 50,
        "rail": 60
    }
]

scenario_results = []

for scenario in scenarios:

    result = optimize_fleet(
        segment_demand,
        total_buses=scenario["buses"],
        total_rail=scenario["rail"]
    )

    result["scenario"] = scenario["scenario"]

    scenario_results.append(result)

scenario_results = pd.DataFrame(
    scenario_results
)

scenario_results = scenario_results[
    [
        "scenario",
        "buses_available",
        "rail_available",
        "buses_used",
        "rail_used",
        "capacity",
        "unmet_demand",
        "coverage_pct",
        "operating_cost"
    ]
]

scenario_results

,scenario,buses_available,rail_available,buses_used,rail_used,capacity,unmet_demand,coverage_pct,operating_cost
0,Current Fleet,40,50,40,50,52000,28789,64.37,440.0
1,+10 Buses,50,50,50,50,52500,28289,64.98,450.0
2,+10 Rail,40,60,40,55,57000,25880,67.97,480.0
3,+20 Buses,60,50,60,50,53000,27789,65.60,460.0
4,+20 Rail,40,70,40,55,57000,25880,67.97,480.0
5,+10 Bus +10 Rail,50,60,50,55,57500,25380,68.58,490.0


In [23]:
# ============================================
# 18. OPTIMIZATION ACROSS ALL TEST DAYS
# ============================================

daily_results = []

for date in sorted(forecast_output["date"].unique()):

    day_data = forecast_output[
        forecast_output["date"] == date
    ].copy()

    # Convert to segment-level mode demand
    day_segment_demand = (
        day_data
        .pivot_table(
            index="segment",
            columns="mode",
            values="predicted_demand",
            aggfunc="sum"
        )
        .fillna(0)
        .reset_index()
    )

    if "Bus" not in day_segment_demand.columns:
        day_segment_demand["Bus"] = 0

    if "Rail" not in day_segment_demand.columns:
        day_segment_demand["Rail"] = 0

    day_segment_demand = day_segment_demand[
        ["segment", "Bus", "Rail"]
    ]

    day_segment_demand.columns = [
        "segment",
        "bus_demand",
        "rail_demand"
    ]

    result = optimize_fleet(
        day_segment_demand,
        total_buses=40,
        total_rail=50
    )

    result["date"] = date

    daily_results.append(result)

daily_results = pd.DataFrame(daily_results)

daily_results = daily_results[
    [
        "date",
        "buses_available",
        "rail_available",
        "buses_used",
        "rail_used",
        "capacity",
        "unmet_demand",
        "coverage_pct",
        "operating_cost"
    ]
]

daily_results

,date,buses_available,rail_available,buses_used,rail_used,capacity,unmet_demand,coverage_pct,operating_cost
0,2025-10-14,40,50,40,50,52000,28789,64.37,440.0
1,2025-10-15,40,50,40,50,52000,28384,64.69,440.0
2,2025-10-16,40,50,40,50,52000,30027,63.39,440.0
3,2025-10-17,40,50,40,50,52000,28833,64.33,440.0
4,2025-10-18,40,50,40,48,50000,24005,66.46,424.0
5,2025-10-19,40,50,40,46,48000,23371,66.28,408.0
6,2025-10-20,40,50,40,50,52000,98864,34.47,440.0
7,2025-10-21,40,50,40,50,52000,30704,62.87,440.0
8,2025-10-22,40,50,40,50,52000,33025,61.16,440.0
9,2025-10-23,40,50,40,50,52000,31087,62.59,440.0


In [24]:
# ============================================
# 19. INVESTIGATE EXTREME DEMAND DAY
# ============================================

extreme_date = daily_results.loc[
    daily_results["unmet_demand"].idxmax(),
    "date"
]

print("Extreme demand date:", extreme_date)

extreme_day = forecast_output[
    forecast_output["date"] == extreme_date
].copy()

print("\nRoute-level demand:")
print(
    extreme_day[
        [
            "route",
            "passengers",
            "predicted_demand",
            "forecast_error"
        ]
    ].to_string(index=False)
)

print("\nTotal actual demand:",
      extreme_day["passengers"].sum())

print("Total predicted demand:",
      extreme_day["predicted_demand"].sum())

Extreme demand date: 2025-10-20 00:00:00

Route-level demand:
                   route  passengers  predicted_demand  forecast_error
      Howrah_Kolkata_Bus       11624             11867            -243
     Howrah_Kolkata_Rail       23359             26119           -2760
 Kharagpur_Midnapore_Bus       17284             15101            2183
Kharagpur_Midnapore_Rail       21237             22259           -1022
  Midnapore_Uluberia_Bus        9590             12117           -2527
 Midnapore_Uluberia_Rail       24873             24846              27
     Uluberia_Howrah_Bus       13929             13796             133
    Uluberia_Howrah_Rail       23112             24759           -1647

Total actual demand: 145008
Total predicted demand: 150864


In [27]:
forecast_output = pd.read_csv(
    "../outputs/demand_forecast.csv"
)

In [28]:
forecast_output.columns.tolist()

['date',
 'route',
 'passengers',
 'festival',
 'weekend',
 'day_of_week',
 'predicted_demand',
 'forecast_error']

In [30]:
# ============================================
# 20. FESTIVAL STATUS OF EXTREME DAY
# ============================================

# Make sure dates use the same format
forecast_output["date"] = pd.to_datetime(forecast_output["date"])
extreme_date = pd.to_datetime(extreme_date)

print("Extreme date:", extreme_date)

festival_check = (
    forecast_output[
        forecast_output["date"] == extreme_date
    ][
        ["date", "festival"]
    ]
    .drop_duplicates()
)

festival_check

Extreme date: 2025-10-20 00:00:00


,date,festival
6,2025-10-20,1


In [31]:
# ============================================
# 21. EXTREME DAY DEMAND ANALYSIS
# ============================================

extreme_day = (
    forecast_output[
        forecast_output["date"] == extreme_date
    ][
        [
            "date",
            "route",
            "passengers",
            "predicted_demand",
            "festival",
            "weekend",
            "forecast_error"
        ]
    ]
    .copy()
)

print("EXTREME DAY:", extreme_date)
print("\nRoute-level demand:")
display(extreme_day)

print("\nTotal actual demand:",
      extreme_day["passengers"].sum())

print("Total predicted demand:",
      extreme_day["predicted_demand"].sum())

print("Festival status:",
      extreme_day["festival"].iloc[0])

print("Weekend status:",
      extreme_day["weekend"].iloc[0])

EXTREME DAY: 2025-10-20 00:00:00

Route-level demand:


,date,route,passengers,predicted_demand,festival,weekend,forecast_error
6,2025-10-20,Howrah_Kolkata_Bus,11624,11867,1,0,-242.844887
22,2025-10-20,Howrah_Kolkata_Rail,23359,26119,1,0,-2760.390580
38,2025-10-20,Kharagpur_Midnapore_Bus,17284,15101,1,0,2182.936875
54,2025-10-20,Kharagpur_Midnapore_Rail,21237,22259,1,0,-1022.427135
70,2025-10-20,Midnapore_Uluberia_Bus,9590,12117,1,0,-2526.710981
86,2025-10-20,Midnapore_Uluberia_Rail,24873,24846,1,0,27.484310
102,2025-10-20,Uluberia_Howrah_Bus,13929,13796,1,0,132.683066
118,2025-10-20,Uluberia_Howrah_Rail,23112,24759,1,0,-1646.752230



Total actual demand: 145008
Total predicted demand: 150864
Festival status: 1
Weekend status: 0


In [32]:
extreme_day[
    [
        "route",
        "passengers",
        "predicted_demand",
        "festival",
        "forecast_error"
    ]
].sort_values(
    "predicted_demand",
    ascending=False
)

,route,passengers,predicted_demand,festival,forecast_error
22,Howrah_Kolkata_Rail,23359,26119,1,-2760.390580
86,Midnapore_Uluberia_Rail,24873,24846,1,27.484310
118,Uluberia_Howrah_Rail,23112,24759,1,-1646.752230
54,Kharagpur_Midnapore_Rail,21237,22259,1,-1022.427135
38,Kharagpur_Midnapore_Bus,17284,15101,1,2182.936875
102,Uluberia_Howrah_Bus,13929,13796,1,132.683066
70,Midnapore_Uluberia_Bus,9590,12117,1,-2526.710981
6,Howrah_Kolkata_Bus,11624,11867,1,-242.844887


In [33]:
# ============================================
# 22. FESTIVAL CAPACITY REQUIREMENT
# ============================================

festival_demand = extreme_day["predicted_demand"].sum()

print("FESTIVAL CAPACITY REQUIREMENT")
print("-----------------------------------")
print(f"Forecast festival demand : {festival_demand:,.0f}")

for target in [0.80, 0.90, 0.95, 1.00]:

    required_capacity = festival_demand * target
    capacity_gap = max(0, required_capacity - 52000)

    print(f"\nTarget coverage: {target:.0%}")
    print(f"Required capacity : {required_capacity:,.0f}")
    print(f"Additional capacity required : {capacity_gap:,.0f}")

FESTIVAL CAPACITY REQUIREMENT
-----------------------------------
Forecast festival demand : 150,864

Target coverage: 80%
Required capacity : 120,691
Additional capacity required : 68,691

Target coverage: 90%
Required capacity : 135,778
Additional capacity required : 83,778

Target coverage: 95%
Required capacity : 143,321
Additional capacity required : 91,321

Target coverage: 100%
Required capacity : 150,864
Additional capacity required : 98,864


In [34]:
# ============================================
# 23. ADDITIONAL RAIL SERVICES REQUIRED
# ============================================

current_capacity = 52000
rail_capacity = 1000

print("ADDITIONAL RAIL SERVICE REQUIREMENT")
print("-----------------------------------")

for target in [0.80, 0.90, 0.95, 1.00]:

    required_capacity = festival_demand * target

    additional_capacity = max(
        0,
        required_capacity - current_capacity
    )

    additional_rail = int(
        np.ceil(additional_capacity / rail_capacity)
    )

    print(
        f"{target:.0%} coverage → "
        f"{additional_rail} additional rail services"
    )

ADDITIONAL RAIL SERVICE REQUIREMENT
-----------------------------------
80% coverage → 69 additional rail services
90% coverage → 84 additional rail services
95% coverage → 92 additional rail services
100% coverage → 99 additional rail services


## 7. Fleet-Based Capacity Optimization

The initial optimization treated available vehicles as directly equivalent
to transport services. This section introduces a fleet-to-service relationship,
where each vehicle can perform multiple services per day.

The model determines the number of bus and rail services required on each
corridor segment while respecting fleet availability, vehicle utilization,
capacity constraints, and unmet-demand penalties.

In [35]:
# ============================================
# 7. FLEET-BASED CAPACITY CONFIGURATION
# ============================================

BUS_FLEET = 40
RAIL_FLEET = 50

BUS_CAPACITY = 50
RAIL_CAPACITY = 1000

BUS_TRIPS_PER_VEHICLE = 10
RAIL_TRIPS_PER_VEHICLE = 2

MAX_BUS_SERVICES = BUS_FLEET * BUS_TRIPS_PER_VEHICLE
MAX_RAIL_SERVICES = RAIL_FLEET * RAIL_TRIPS_PER_VEHICLE

BUS_COST = 1
RAIL_COST = 8
UNMET_PENALTY = 20

print("Fleet-based capacity configuration")
print("-----------------------------------")
print(f"Bus fleet                 : {BUS_FLEET}")
print(f"Rail fleet                : {RAIL_FLEET}")
print(f"Bus capacity/service      : {BUS_CAPACITY}")
print(f"Rail capacity/service     : {RAIL_CAPACITY}")
print(f"Bus trips/vehicle/day     : {BUS_TRIPS_PER_VEHICLE}")
print(f"Rail trips/vehicle/day    : {RAIL_TRIPS_PER_VEHICLE}")
print(f"Maximum bus services/day  : {MAX_BUS_SERVICES}")
print(f"Maximum rail services/day : {MAX_RAIL_SERVICES}")

Fleet-based capacity configuration
-----------------------------------
Bus fleet                 : 40
Rail fleet                : 50
Bus capacity/service      : 50
Rail capacity/service     : 1000
Bus trips/vehicle/day     : 10
Rail trips/vehicle/day    : 2
Maximum bus services/day  : 400
Maximum rail services/day : 100


In [36]:
# ============================================
# 8. FLEET-BASED NETWORK CAPACITY
# ============================================

bus_network_capacity = MAX_BUS_SERVICES * BUS_CAPACITY
rail_network_capacity = MAX_RAIL_SERVICES * RAIL_CAPACITY

total_network_capacity = (
    bus_network_capacity +
    rail_network_capacity
)

print("FLEET-BASED NETWORK CAPACITY")
print("-----------------------------------")
print(f"Maximum bus capacity  : {bus_network_capacity:,.0f}")
print(f"Maximum rail capacity : {rail_network_capacity:,.0f}")
print(f"Total capacity        : {total_network_capacity:,.0f}")

FLEET-BASED NETWORK CAPACITY
-----------------------------------
Maximum bus capacity  : 20,000
Maximum rail capacity : 100,000
Total capacity        : 120,000


In [37]:
# ============================================
# 9. FLEET-BASED OPTIMIZATION MODEL
# ============================================

fleet_model = pulp.LpProblem(
    "Fleet_Based_Multimodal_Optimization",
    pulp.LpMinimize
)

segments = segment_demand["segment"].tolist()

# --------------------------------------------
# Decision variables
# --------------------------------------------

bus_services = {
    s: pulp.LpVariable(
        f"bus_services_{s}",
        lowBound=0,
        cat="Integer"
    )
    for s in segments
}

rail_services = {
    s: pulp.LpVariable(
        f"rail_services_{s}",
        lowBound=0,
        cat="Integer"
    )
    for s in segments
}

unmet_bus = {
    s: pulp.LpVariable(
        f"unmet_bus_{s}",
        lowBound=0,
        cat="Continuous"
    )
    for s in segments
}

unmet_rail = {
    s: pulp.LpVariable(
        f"unmet_rail_{s}",
        lowBound=0,
        cat="Continuous"
    )
    for s in segments
}

In [38]:
# ============================================
# 10. OBJECTIVE FUNCTION
# ============================================

fleet_model += pulp.lpSum(
    BUS_COST * bus_services[s]
    + RAIL_COST * rail_services[s]
    + UNMET_PENALTY * (
        unmet_bus[s] + unmet_rail[s]
    )
    for s in segments
)

In [39]:
# ============================================
# 11. DEMAND CONSTRAINTS
# ============================================

for _, row in segment_demand.iterrows():

    s = row["segment"]

    # Bus demand
    fleet_model += (
        BUS_CAPACITY * bus_services[s]
        + unmet_bus[s]
        >= row["bus_demand"]
    )

    # Rail demand
    fleet_model += (
        RAIL_CAPACITY * rail_services[s]
        + unmet_rail[s]
        >= row["rail_demand"]
    )

In [40]:
# ============================================
# 12. FLEET / SERVICE CONSTRAINTS
# ============================================

# Total bus services cannot exceed
# fleet × trips per vehicle
fleet_model += pulp.lpSum(
    bus_services[s]
    for s in segments
) <= MAX_BUS_SERVICES

# Total rail services cannot exceed
# fleet × trips per vehicle
fleet_model += pulp.lpSum(
    rail_services[s]
    for s in segments
) <= MAX_RAIL_SERVICES

In [41]:
# ============================================
# 13. SOLVE FLEET-BASED MODEL
# ============================================

fleet_model.solve(
    pulp.PULP_CBC_CMD(msg=False)
)

print(
    "Fleet-based optimization status:",
    pulp.LpStatus[fleet_model.status]
)

Fleet-based optimization status: Optimal


In [42]:
# ============================================
# 14. FLEET-BASED OPTIMIZATION RESULTS
# ============================================

fleet_results = []

for _, row in segment_demand.iterrows():

    s = row["segment"]

    bus = int(round(
        pulp.value(bus_services[s])
    ))

    rail = int(round(
        pulp.value(rail_services[s])
    ))

    bus_capacity = bus * BUS_CAPACITY
    rail_capacity = rail * RAIL_CAPACITY

    bus_unmet_value = pulp.value(
        unmet_bus[s]
    )

    rail_unmet_value = pulp.value(
        unmet_rail[s]
    )

    fleet_results.append({
        "segment": s,
        "bus_demand": round(row["bus_demand"]),
        "rail_demand": round(row["rail_demand"]),
        "bus_services": bus,
        "rail_services": rail,
        "bus_capacity": bus_capacity,
        "rail_capacity": rail_capacity,
        "bus_unmet": round(bus_unmet_value, 2),
        "rail_unmet": round(rail_unmet_value, 2)
    })

fleet_results = pd.DataFrame(fleet_results)

fleet_results

,segment,bus_demand,rail_demand,bus_services,rail_services,bus_capacity,rail_capacity,bus_unmet,rail_unmet
0,Howrah → Kolkata,6227,14524,0,15,0,15000,6227.0,0.0
1,Kharagpur → Midnapore,8462,11205,169,12,8450,12000,12.0,0.0
2,Midnapore → Uluberia,6568,14519,99,15,4950,15000,1618.0,0.0
3,Uluberia → Howrah,6623,12661,132,13,6600,13000,23.0,0.0


In [43]:
# ============================================
# 15. FLEET-BASED NETWORK SUMMARY
# ============================================

total_bus_demand = fleet_results["bus_demand"].sum()
total_rail_demand = fleet_results["rail_demand"].sum()

total_demand = (
    total_bus_demand +
    total_rail_demand
)

total_bus_capacity = (
    fleet_results["bus_capacity"].sum()
)

total_rail_capacity = (
    fleet_results["rail_capacity"].sum()
)

total_capacity = (
    total_bus_capacity +
    total_rail_capacity
)

total_bus_unmet = (
    fleet_results["bus_unmet"].sum()
)

total_rail_unmet = (
    fleet_results["rail_unmet"].sum()
)

total_unmet = (
    total_bus_unmet +
    total_rail_unmet
)

coverage = (
    (total_demand - total_unmet)
    / total_demand
    * 100
)

buses_used = fleet_results["bus_services"].sum()
rail_used = fleet_results["rail_services"].sum()

operating_cost = (
    buses_used * BUS_COST +
    rail_used * RAIL_COST
)

print("FLEET-BASED NETWORK SUMMARY")
print("-----------------------------------")
print(f"Total demand          : {total_demand:,.0f}")
print(f"Bus demand            : {total_bus_demand:,.0f}")
print(f"Rail demand           : {total_rail_demand:,.0f}")
print()
print(f"Bus services used     : {buses_used}")
print(f"Maximum bus services  : {MAX_BUS_SERVICES}")
print(f"Rail services used    : {rail_used}")
print(f"Maximum rail services : {MAX_RAIL_SERVICES}")
print()
print(f"Bus capacity          : {total_bus_capacity:,.0f}")
print(f"Rail capacity         : {total_rail_capacity:,.0f}")
print(f"Total capacity        : {total_capacity:,.0f}")
print()
print(f"Bus unmet demand      : {total_bus_unmet:,.0f}")
print(f"Rail unmet demand     : {total_rail_unmet:,.0f}")
print(f"Total unmet demand    : {total_unmet:,.0f}")
print()
print(f"Demand coverage       : {coverage:.2f}%")
print(f"Operating cost        : {operating_cost:.2f}")

FLEET-BASED NETWORK SUMMARY
-----------------------------------
Total demand          : 80,789
Bus demand            : 27,880
Rail demand           : 52,909

Bus services used     : 400
Maximum bus services  : 400
Rail services used    : 55
Maximum rail services : 100

Bus capacity          : 20,000
Rail capacity         : 55,000
Total capacity        : 75,000

Bus unmet demand      : 7,880
Rail unmet demand     : 0
Total unmet demand    : 7,880

Demand coverage       : 90.25%
Operating cost        : 840.00


In [45]:
# ============================================
# 16. FESTIVAL DAY DEMAND
# ============================================

# Make sure date is datetime
forecast_output["date"] = pd.to_datetime(
    forecast_output["date"]
)

# Recreate mode from route name
forecast_output["mode"] = (
    forecast_output["route"]
    .str.rsplit("_", n=1)
    .str[-1]
)

# Recreate origin and destination
route_parts = forecast_output["route"].str.rsplit(
    "_", n=2, expand=True
)

forecast_output["origin"] = route_parts[0]
forecast_output["destination"] = route_parts[1]

# Create physical corridor segment
forecast_output["segment"] = (
    forecast_output["origin"]
    + " → "
    + forecast_output["destination"]
)

# Select festival / extreme day
festival_date = pd.to_datetime(extreme_date)

festival_data = forecast_output[
    forecast_output["date"] == festival_date
].copy()

# Aggregate demand by physical segment and mode
festival_segment_demand = (
    festival_data
    .pivot_table(
        index="segment",
        columns="mode",
        values="predicted_demand",
        aggfunc="sum"
    )
    .fillna(0)
    .reset_index()
)

# Ensure both modes exist
if "Bus" not in festival_segment_demand.columns:
    festival_segment_demand["Bus"] = 0

if "Rail" not in festival_segment_demand.columns:
    festival_segment_demand["Rail"] = 0

festival_segment_demand = festival_segment_demand[
    ["segment", "Bus", "Rail"]
]

festival_segment_demand.columns = [
    "segment",
    "bus_demand",
    "rail_demand"
]

print("Festival date:", festival_date.date())
print("\nFestival segment demand:")
display(festival_segment_demand)

print(
    "\nTotal festival forecast demand:",
    int(festival_segment_demand["bus_demand"].sum()
        + festival_segment_demand["rail_demand"].sum())
)

Festival date: 2025-10-20

Festival segment demand:


,segment,bus_demand,rail_demand
0,Howrah → Kolkata,11867,26119
1,Kharagpur → Midnapore,15101,22259
2,Midnapore → Uluberia,12117,24846
3,Uluberia → Howrah,13796,24759



Total festival forecast demand: 150864


In [46]:
# ============================================
# 17. FLEET-BASED OPTIMIZATION FUNCTION
# ============================================

def optimize_fleet_based(
    demand_data,
    max_bus_services=MAX_BUS_SERVICES,
    max_rail_services=MAX_RAIL_SERVICES
):
    """
    Fleet-based multimodal optimization.

    Bus demand can only be served by bus services.
    Rail demand can only be served by rail services.

    The available number of services is determined by:
        fleet size × trips per vehicle per day
    """

    model = pulp.LpProblem(
        "Fleet_Based_Transport_Allocation",
        pulp.LpMinimize
    )

    segments = demand_data["segment"].tolist()

    # ----------------------------------------
    # Decision variables
    # ----------------------------------------

    bus_services = {
        s: pulp.LpVariable(
            f"bus_service_{s}",
            lowBound=0,
            cat="Integer"
        )
        for s in segments
    }

    rail_services = {
        s: pulp.LpVariable(
            f"rail_service_{s}",
            lowBound=0,
            cat="Integer"
        )
        for s in segments
    }

    unmet_bus = {
        s: pulp.LpVariable(
            f"unmet_bus_{s}",
            lowBound=0
        )
        for s in segments
    }

    unmet_rail = {
        s: pulp.LpVariable(
            f"unmet_rail_{s}",
            lowBound=0
        )
        for s in segments
    }

    # ----------------------------------------
    # Objective
    # ----------------------------------------

    model += pulp.lpSum(
        BUS_COST * bus_services[s]
        + RAIL_COST * rail_services[s]
        + UNMET_PENALTY *
        (unmet_bus[s] + unmet_rail[s])
        for s in segments
    )

    # ----------------------------------------
    # Demand constraints
    # ----------------------------------------

    for _, row in demand_data.iterrows():

        s = row["segment"]

        model += (
            BUS_CAPACITY * bus_services[s]
            + unmet_bus[s]
            >= row["bus_demand"]
        )

        model += (
            RAIL_CAPACITY * rail_services[s]
            + unmet_rail[s]
            >= row["rail_demand"]
        )

    # ----------------------------------------
    # Network service constraints
    # ----------------------------------------

    model += pulp.lpSum(
        bus_services[s]
        for s in segments
    ) <= max_bus_services

    model += pulp.lpSum(
        rail_services[s]
        for s in segments
    ) <= max_rail_services

    # ----------------------------------------
    # Solve
    # ----------------------------------------

    model.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    # ----------------------------------------
    # Extract results
    # ----------------------------------------

    results = []

    for _, row in demand_data.iterrows():

        s = row["segment"]

        buses = int(round(
            pulp.value(bus_services[s])
        ))

        rail = int(round(
            pulp.value(rail_services[s])
        ))

        bus_capacity = (
            buses * BUS_CAPACITY
        )

        rail_capacity = (
            rail * RAIL_CAPACITY
        )

        bus_unmet_value = pulp.value(
            unmet_bus[s]
        )

        rail_unmet_value = pulp.value(
            unmet_rail[s]
        )

        results.append({
            "segment": s,
            "bus_demand": round(row["bus_demand"]),
            "rail_demand": round(row["rail_demand"]),
            "bus_services": buses,
            "rail_services": rail,
            "bus_capacity": bus_capacity,
            "rail_capacity": rail_capacity,
            "bus_unmet": round(
                bus_unmet_value, 2
            ),
            "rail_unmet": round(
                rail_unmet_value, 2
            )
        })

    results = pd.DataFrame(results)

    # ----------------------------------------
    # Network summary
    # ----------------------------------------

    total_demand = (
        results["bus_demand"].sum()
        + results["rail_demand"].sum()
    )

    total_unmet = (
        results["bus_unmet"].sum()
        + results["rail_unmet"].sum()
    )

    total_capacity = (
        results["bus_capacity"].sum()
        + results["rail_capacity"].sum()
    )

    coverage = (
        (total_demand - total_unmet)
        / total_demand
        * 100
    )

    summary = {
        "status": pulp.LpStatus[model.status],
        "bus_services_used":
            results["bus_services"].sum(),
        "rail_services_used":
            results["rail_services"].sum(),
        "capacity":
            total_capacity,
        "unmet_demand":
            round(total_unmet),
        "coverage_pct":
            round(coverage, 2),
        "operating_cost":
            round(
                results["bus_services"].sum()
                * BUS_COST
                +
                results["rail_services"].sum()
                * RAIL_COST,
                2
            )
    }

    return results, summary

In [47]:
# ============================================
# 18. FESTIVAL DAY OPTIMIZATION
# ============================================

festival_results, festival_summary = (
    optimize_fleet_based(
        festival_segment_demand
    )
)

print("FESTIVAL OPTIMIZATION SUMMARY")
print("-----------------------------------")

for key, value in festival_summary.items():
    print(f"{key}: {value}")

print("\nRoute-level allocation:")
display(festival_results)

FESTIVAL OPTIMIZATION SUMMARY
-----------------------------------
status: Optimal
bus_services_used: 400
rail_services_used: 100
capacity: 120000
unmet_demand: 32881
coverage_pct: 78.2
operating_cost: 1200

Route-level allocation:


,segment,bus_demand,rail_demand,bus_services,rail_services,bus_capacity,rail_capacity,bus_unmet,rail_unmet
0,Howrah → Kolkata,11867,26119,0,27,0,27000,11867.0,0.0
1,Kharagpur → Midnapore,15101,22259,302,23,15100,23000,1.0,0.0
2,Midnapore → Uluberia,12117,24846,0,25,0,25000,12117.0,0.0
3,Uluberia → Howrah,13796,24759,98,25,4900,25000,8896.0,0.0


## 19. Modal Substitution Analysis

During extreme-demand periods, passengers unable to obtain their
preferred transport mode may potentially shift to an alternative mode.

This section evaluates different modal-substitution assumptions to
determine whether spare rail capacity can reduce unmet bus demand.

Substitution scenarios of 0%, 10%, 20%, 30%, and 40% are evaluated.

In [48]:
# ============================================
# 19. MODAL SUBSTITUTION OPTIMIZATION
# ============================================

def optimize_with_modal_substitution(
    demand_data,
    max_bus_services=MAX_BUS_SERVICES,
    max_rail_services=MAX_RAIL_SERVICES,
    max_modal_shift=0.0
):
    """
    Multimodal fleet optimization with optional
    bus-to-rail passenger substitution.

    max_modal_shift:
        Maximum fraction of bus demand that may
        be shifted to rail.
    """

    model = pulp.LpProblem(
        "Modal_Substitution_Optimization",
        pulp.LpMinimize
    )

    segments = demand_data["segment"].tolist()

    # ----------------------------------------
    # Decision variables
    # ----------------------------------------

    bus_services = {
        s: pulp.LpVariable(
            f"bus_service_{s}",
            lowBound=0,
            cat="Integer"
        )
        for s in segments
    }

    rail_services = {
        s: pulp.LpVariable(
            f"rail_service_{s}",
            lowBound=0,
            cat="Integer"
        )
        for s in segments
    }

    unmet_bus = {
        s: pulp.LpVariable(
            f"unmet_bus_{s}",
            lowBound=0
        )
        for s in segments
    }

    unmet_rail = {
        s: pulp.LpVariable(
            f"unmet_rail_{s}",
            lowBound=0
        )
        for s in segments
    }

    shifted_bus_to_rail = {
        s: pulp.LpVariable(
            f"shifted_bus_to_rail_{s}",
            lowBound=0
        )
        for s in segments
    }

    # ----------------------------------------
    # Objective
    # ----------------------------------------

    model += pulp.lpSum(
        BUS_COST * bus_services[s]
        + RAIL_COST * rail_services[s]
        + UNMET_PENALTY *
        (unmet_bus[s] + unmet_rail[s])
        for s in segments
    )

    # ----------------------------------------
    # Constraints
    # ----------------------------------------

    for _, row in demand_data.iterrows():

        s = row["segment"]

        bus_demand = row["bus_demand"]
        rail_demand = row["rail_demand"]

        # Bus demand balance
        model += (
            BUS_CAPACITY * bus_services[s]
            + unmet_bus[s]
            + shifted_bus_to_rail[s]
            >= bus_demand
        )

        # Rail demand + shifted bus passengers
        model += (
            RAIL_CAPACITY * rail_services[s]
            + unmet_rail[s]
            >=
            rail_demand
            + shifted_bus_to_rail[s]
        )

        # Maximum allowable modal shift
        model += (
            shifted_bus_to_rail[s]
            <= max_modal_shift * bus_demand
        )

    # Fleet/service limits
    model += pulp.lpSum(
        bus_services[s]
        for s in segments
    ) <= max_bus_services

    model += pulp.lpSum(
        rail_services[s]
        for s in segments
    ) <= max_rail_services

    # Solve
    model.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    # ----------------------------------------
    # Extract results
    # ----------------------------------------

    results = []

    for _, row in demand_data.iterrows():

        s = row["segment"]

        buses = int(round(
            pulp.value(bus_services[s])
        ))

        rail = int(round(
            pulp.value(rail_services[s])
        ))

        bus_unmet_value = pulp.value(
            unmet_bus[s]
        )

        rail_unmet_value = pulp.value(
            unmet_rail[s]
        )

        shifted_value = pulp.value(
            shifted_bus_to_rail[s]
        )

        results.append({
            "segment": s,
            "bus_demand": round(row["bus_demand"]),
            "rail_demand": round(row["rail_demand"]),
            "bus_services": buses,
            "rail_services": rail,
            "bus_unmet": round(
                bus_unmet_value, 2
            ),
            "rail_unmet": round(
                rail_unmet_value, 2
            ),
            "bus_to_rail_shift": round(
                shifted_value, 2
            )
        })

    results = pd.DataFrame(results)

    total_demand = (
        results["bus_demand"].sum()
        + results["rail_demand"].sum()
    )

    total_unmet = (
        results["bus_unmet"].sum()
        + results["rail_unmet"].sum()
    )

    total_shifted = (
        results["bus_to_rail_shift"].sum()
    )

    coverage = (
        (total_demand - total_unmet)
        / total_demand
        * 100
    )

    total_bus_used = results["bus_services"].sum()
    total_rail_used = results["rail_services"].sum()

    capacity = (
        total_bus_used * BUS_CAPACITY
        + total_rail_used * RAIL_CAPACITY
    )

    operating_cost = (
        total_bus_used * BUS_COST
        + total_rail_used * RAIL_COST
    )

    summary = {
        "modal_shift_limit": max_modal_shift,
        "status": pulp.LpStatus[model.status],
        "bus_services_used": total_bus_used,
        "rail_services_used": total_rail_used,
        "capacity": capacity,
        "passengers_shifted": round(total_shifted),
        "unmet_demand": round(total_unmet),
        "coverage_pct": round(coverage, 2),
        "operating_cost": round(
            operating_cost, 2
        )
    }

    return results, summary

In [49]:
# ============================================
# 20. ZERO MODAL SUBSTITUTION
# ============================================

no_shift_results, no_shift_summary = (
    optimize_with_modal_substitution(
        festival_segment_demand,
        max_modal_shift=0.0
    )
)

print("0% MODAL SUBSTITUTION")
print("-----------------------------------")

for key, value in no_shift_summary.items():
    print(f"{key}: {value}")

0% MODAL SUBSTITUTION
-----------------------------------
modal_shift_limit: 0.0
status: Optimal
bus_services_used: 400
rail_services_used: 100
capacity: 120000
passengers_shifted: 0
unmet_demand: 32881
coverage_pct: 78.2
operating_cost: 1200


In [50]:
# ============================================
# 21. MODAL SUBSTITUTION SCENARIOS
# ============================================

shift_levels = [0.0, 0.10, 0.20, 0.30, 0.40]

modal_results = []

for shift in shift_levels:

    _, summary = optimize_with_modal_substitution(
        festival_segment_demand,
        max_modal_shift=shift
    )

    modal_results.append(summary)

modal_results = pd.DataFrame(
    modal_results
)

modal_results

,modal_shift_limit,status,bus_services_used,rail_services_used,capacity,passengers_shifted,unmet_demand,coverage_pct,operating_cost
0,0.0,Optimal,400,100,120000,0,32881,78.20,1200
1,0.1,Optimal,400,100,120000,2395,30864,79.54,1200
2,0.2,Optimal,400,100,120000,2395,30864,79.54,1200
3,0.3,Optimal,400,100,120000,2395,30864,79.54,1200
4,0.4,Optimal,400,100,120000,2395,30864,79.54,1200


In [51]:
# ============================================
# 22. FESTIVAL MODAL SHIFT DETAIL
# ============================================

shift_results, shift_summary = (
    optimize_with_modal_substitution(
        festival_segment_demand,
        max_modal_shift=0.10
    )
)

print("10% MODAL SHIFT DETAIL")
print("-----------------------------------")

display(
    shift_results[
        [
            "segment",
            "bus_demand",
            "rail_demand",
            "bus_services",
            "rail_services",
            "bus_unmet",
            "rail_unmet",
            "bus_to_rail_shift"
        ]
    ]
)

10% MODAL SHIFT DETAIL
-----------------------------------


,segment,bus_demand,rail_demand,bus_services,rail_services,bus_unmet,rail_unmet,bus_to_rail_shift
0,Howrah → Kolkata,11867,26119,0,26,11867.0,119.0,0.0
1,Kharagpur → Midnapore,15101,22259,302,22,1.0,259.0,0.0
2,Midnapore → Uluberia,12117,24846,0,26,10963.0,0.0,1154.0
3,Uluberia → Howrah,13796,24759,98,26,7655.0,0.0,1241.0


In [52]:
# ============================================
# 23. FESTIVAL FLEET EXPANSION SCENARIOS
# ============================================

fleet_scenarios = [
    {"scenario": "Current Fleet", "bus_fleet": 40, "rail_fleet": 50},
    {"scenario": "+10 Buses", "bus_fleet": 50, "rail_fleet": 50},
    {"scenario": "+20 Buses", "bus_fleet": 60, "rail_fleet": 50},
    {"scenario": "+30 Buses", "bus_fleet": 70, "rail_fleet": 50},
    {"scenario": "+10 Rail", "bus_fleet": 40, "rail_fleet": 60},
    {"scenario": "+20 Rail", "bus_fleet": 40, "rail_fleet": 70},
    {"scenario": "+30 Rail", "bus_fleet": 40, "rail_fleet": 80},
    {"scenario": "+10 Bus +10 Rail", "bus_fleet": 50, "rail_fleet": 60},
    {"scenario": "+20 Bus +20 Rail", "bus_fleet": 60, "rail_fleet": 70},
    {"scenario": "+30 Bus +30 Rail", "bus_fleet": 70, "rail_fleet": 80},
]

scenario_results = []

for scenario in fleet_scenarios:

    max_bus_services = (
        scenario["bus_fleet"]
        * BUS_TRIPS_PER_VEHICLE
    )

    max_rail_services = (
        scenario["rail_fleet"]
        * RAIL_TRIPS_PER_VEHICLE
    )

    _, summary = optimize_with_modal_substitution(
        festival_segment_demand,
        max_bus_services=max_bus_services,
        max_rail_services=max_rail_services,
        max_modal_shift=0.10
    )

    scenario_results.append({
        "scenario": scenario["scenario"],
        "bus_fleet": scenario["bus_fleet"],
        "rail_fleet": scenario["rail_fleet"],
        "bus_services_available": max_bus_services,
        "rail_services_available": max_rail_services,
        "capacity": summary["capacity"],
        "passengers_shifted": summary["passengers_shifted"],
        "unmet_demand": summary["unmet_demand"],
        "coverage_pct": summary["coverage_pct"],
        "operating_cost": summary["operating_cost"]
    })

fleet_scenario_results = pd.DataFrame(
    scenario_results
)

fleet_scenario_results

,scenario,bus_fleet,rail_fleet,bus_services_available,rail_services_available,capacity,passengers_shifted,unmet_demand,coverage_pct,operating_cost
0,Current Fleet,40,50,400,100,120000,2395,30864,79.54,1200
1,+10 Buses,50,50,500,100,125000,2395,25864,82.86,1300
2,+20 Buses,60,50,600,100,130000,2534,20864,86.17,1400
3,+30 Buses,70,50,700,100,135000,2534,15864,89.48,1500
4,+10 Rail,40,60,400,120,126000,5288,27593,81.71,1248
5,+20 Rail,40,70,400,140,126000,5288,27593,81.71,1248
6,+30 Rail,40,80,400,160,126000,5288,27593,81.71,1248
7,+10 Bus +10 Rail,50,60,500,120,131000,5288,22593,85.02,1348
8,+20 Bus +20 Rail,60,70,600,140,136000,5288,17593,88.34,1448
9,+30 Bus +30 Rail,70,80,700,160,141000,5288,12593,91.65,1548


In [53]:
# ============================================
# 24. FLEET EXPANSION COMPARISON
# ============================================

fleet_scenario_results = (
    fleet_scenario_results
    .sort_values(
        "coverage_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

fleet_scenario_results

,scenario,bus_fleet,rail_fleet,bus_services_available,rail_services_available,capacity,passengers_shifted,unmet_demand,coverage_pct,operating_cost
0,+30 Bus +30 Rail,70,80,700,160,141000,5288,12593,91.65,1548
1,+30 Buses,70,50,700,100,135000,2534,15864,89.48,1500
2,+20 Bus +20 Rail,60,70,600,140,136000,5288,17593,88.34,1448
3,+20 Buses,60,50,600,100,130000,2534,20864,86.17,1400
4,+10 Bus +10 Rail,50,60,500,120,131000,5288,22593,85.02,1348
5,+10 Buses,50,50,500,100,125000,2395,25864,82.86,1300
6,+10 Rail,40,60,400,120,126000,5288,27593,81.71,1248
7,+20 Rail,40,70,400,140,126000,5288,27593,81.71,1248
8,+30 Rail,40,80,400,160,126000,5288,27593,81.71,1248
9,Current Fleet,40,50,400,100,120000,2395,30864,79.54,1200


In [54]:
# ============================================
# 25. TARGET COVERAGE ANALYSIS
# ============================================

targets = [80, 90, 95, 100]

for target in targets:

    feasible = fleet_scenario_results[
        fleet_scenario_results["coverage_pct"] >= target
    ]

    print("\n" + "=" * 50)
    print(f"TARGET COVERAGE: {target}%")
    print("=" * 50)

    if len(feasible) == 0:

        print("No tested scenario achieves this target.")

    else:

        best = feasible.sort_values(
            "operating_cost"
        ).iloc[0]

        print(
            f"Recommended scenario : "
            f"{best['scenario']}"
        )

        print(
            f"Bus fleet             : "
            f"{int(best['bus_fleet'])}"
        )

        print(
            f"Rail fleet            : "
            f"{int(best['rail_fleet'])}"
        )

        print(
            f"Coverage              : "
            f"{best['coverage_pct']:.2f}%"
        )

        print(
            f"Unmet demand          : "
            f"{best['unmet_demand']:,.0f}"
        )

        print(
            f"Operating cost        : "
            f"{best['operating_cost']:.2f}"
        )


TARGET COVERAGE: 80%
Recommended scenario : +10 Rail
Bus fleet             : 40
Rail fleet            : 60
Coverage              : 81.71%
Unmet demand          : 27,593
Operating cost        : 1248.00

TARGET COVERAGE: 90%
Recommended scenario : +30 Bus +30 Rail
Bus fleet             : 70
Rail fleet            : 80
Coverage              : 91.65%
Unmet demand          : 12,593
Operating cost        : 1548.00

TARGET COVERAGE: 95%
No tested scenario achieves this target.

TARGET COVERAGE: 100%
No tested scenario achieves this target.


In [57]:
# ============================================
# 26. SYSTEMATIC FLEET EXPANSION SEARCH
# ============================================

expansion_results = []

for additional_buses in range(0, 101):

    for additional_rail in range(0, 51):

        bus_fleet = BUS_FLEET + additional_buses
        rail_fleet = RAIL_FLEET + additional_rail

        max_bus_services = (
            bus_fleet * BUS_TRIPS_PER_VEHICLE
        )

        max_rail_services = (
            rail_fleet * RAIL_TRIPS_PER_VEHICLE
        )

        _, summary = optimize_with_modal_substitution(
            festival_segment_demand,
            max_bus_services=max_bus_services,
            max_rail_services=max_rail_services,
            max_modal_shift=0.10
        )

        expansion_results.append({
            "additional_buses": additional_buses,
            "additional_rail": additional_rail,
            "bus_fleet": bus_fleet,
            "rail_fleet": rail_fleet,
            "coverage_pct": summary["coverage_pct"],
            "unmet_demand": summary["unmet_demand"],
            "operating_cost": summary["operating_cost"],
            "passengers_shifted": summary["passengers_shifted"]
        })

expansion_results = pd.DataFrame(
    expansion_results
)

print(
    "Scenarios evaluated:",
    len(expansion_results)
)

expansion_results.head()

Scenarios evaluated: 5151


,additional_buses,additional_rail,bus_fleet,rail_fleet,coverage_pct,unmet_demand,operating_cost,passengers_shifted
0,0,0,40,50,79.54,30864,1200,2395
1,0,1,40,51,80.87,28864,1216,4380
2,0,2,40,52,81.58,27789,1232,5288
3,0,3,40,53,81.71,27593,1248,5288
4,0,4,40,54,81.71,27593,1248,5288


In [58]:
# ============================================
# 27. MINIMUM-COST FLEET SOLUTIONS
# ============================================

targets = [80, 90, 95, 100]

best_solutions = []

for target in targets:

    feasible = expansion_results[
        expansion_results["coverage_pct"] >= target
    ].copy()

    if feasible.empty:

        best_solutions.append({
            "target_coverage": target,
            "status": "Not achieved"
        })

    else:

        best = feasible.sort_values(
            [
                "operating_cost",
                "additional_buses",
                "additional_rail"
            ]
        ).iloc[0]

        best_solutions.append({
            "target_coverage": target,
            "status": "Achieved",
            "additional_buses":
                int(best["additional_buses"]),
            "additional_rail":
                int(best["additional_rail"]),
            "final_bus_fleet":
                int(best["bus_fleet"]),
            "final_rail_fleet":
                int(best["rail_fleet"]),
            "coverage_pct":
                best["coverage_pct"],
            "unmet_demand":
                int(best["unmet_demand"]),
            "operating_cost":
                best["operating_cost"]
        })

best_solutions = pd.DataFrame(
    best_solutions
)

best_solutions

,target_coverage,status,additional_buses,additional_rail,final_bus_fleet,final_rail_fleet,coverage_pct,unmet_demand,operating_cost
0,80,Achieved,0,1,40,51,80.87,28864,1216.0
1,90,Achieved,26,2,66,52,90.20,14789,1492.0
2,95,Achieved,41,2,81,52,95.17,7289,1642.0
3,100,Achieved,57,2,97,52,100.00,0,1788.0


## 28. Transport Supply Disruption Analysis

The system is evaluated under partial loss of transportation service capacity.

A disruption is represented as a percentage reduction in available
bus or rail services. The demand forecast remains unchanged, while
the available transportation capacity is reduced.

The optimization model is then re-solved to determine the resulting
unmet demand and demand coverage.

In [59]:
# ============================================
# 28. DISRUPTION-AWARE OPTIMIZATION
# ============================================

def optimize_under_disruption(
    demand_data,
    bus_disruption=0.0,
    rail_disruption=0.0,
    max_modal_shift=0.10
):
    """
    Optimize transportation allocation under
    partial bus/rail service disruption.

    Disruption values are fractions:
        0.10 = 10% service loss
        0.20 = 20% service loss
        etc.
    """

    # Normal maximum services
    normal_bus_services = (
        BUS_FLEET * BUS_TRIPS_PER_VEHICLE
    )

    normal_rail_services = (
        RAIL_FLEET * RAIL_TRIPS_PER_VEHICLE
    )

    # Reduce available services due to disruption
    disrupted_bus_services = int(
        normal_bus_services *
        (1 - bus_disruption)
    )

    disrupted_rail_services = int(
        normal_rail_services *
        (1 - rail_disruption)
    )

    # Reuse our modal-substitution optimizer
    results, summary = optimize_with_modal_substitution(
        demand_data,
        max_bus_services=disrupted_bus_services,
        max_rail_services=disrupted_rail_services,
        max_modal_shift=max_modal_shift
    )

    # Add disruption information
    summary["bus_disruption_pct"] = (
        bus_disruption * 100
    )

    summary["rail_disruption_pct"] = (
        rail_disruption * 100
    )

    summary["bus_services_available"] = (
        disrupted_bus_services
    )

    summary["rail_services_available"] = (
        disrupted_rail_services
    )

    return results, summary

In [60]:
# ============================================
# 29. FESTIVAL BASELINE
# ============================================

baseline_results, baseline_summary = (
    optimize_under_disruption(
        festival_segment_demand,
        bus_disruption=0.0,
        rail_disruption=0.0,
        max_modal_shift=0.10
    )
)

print("FESTIVAL BASELINE")
print("-----------------------------------")

for key, value in baseline_summary.items():
    print(f"{key}: {value}")

FESTIVAL BASELINE
-----------------------------------
modal_shift_limit: 0.1
status: Optimal
bus_services_used: 400
rail_services_used: 100
capacity: 120000
passengers_shifted: 2395
unmet_demand: 30864
coverage_pct: 79.54
operating_cost: 1200
bus_disruption_pct: 0.0
rail_disruption_pct: 0.0
bus_services_available: 400
rail_services_available: 100


In [61]:
# ============================================
# 30. BUS DISRUPTION SCENARIOS
# ============================================

bus_disruption_levels = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30
]

bus_disruption_results = []

for disruption in bus_disruption_levels:

    _, summary = optimize_under_disruption(
        festival_segment_demand,
        bus_disruption=disruption,
        rail_disruption=0.0,
        max_modal_shift=0.10
    )

    bus_disruption_results.append({
        "bus_disruption_pct":
            disruption * 100,

        "bus_services_available":
            summary["bus_services_available"],

        "rail_services_available":
            summary["rail_services_available"],

        "capacity":
            summary["capacity"],

        "passengers_shifted":
            summary["passengers_shifted"],

        "unmet_demand":
            summary["unmet_demand"],

        "coverage_pct":
            summary["coverage_pct"],

        "operating_cost":
            summary["operating_cost"]
    })

bus_disruption_results = pd.DataFrame(
    bus_disruption_results
)

bus_disruption_results

,bus_disruption_pct,bus_services_available,rail_services_available,capacity,passengers_shifted,unmet_demand,coverage_pct,operating_cost
0,0.0,400,100,120000,2395,30864,79.54,1200
1,5.0,380,100,119000,2395,31864,78.88,1180
2,10.0,360,100,118000,2395,32864,78.22,1160
3,15.0,340,100,117000,2395,33864,77.55,1140
4,20.0,320,100,116000,2395,34864,76.89,1120
5,25.0,300,100,115000,2395,35864,76.23,1100
6,30.0,280,100,114000,2395,36864,75.56,1080


In [62]:
# ============================================
# 31. RAIL DISRUPTION SCENARIOS
# ============================================

rail_disruption_levels = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30
]

rail_disruption_results = []

for disruption in rail_disruption_levels:

    _, summary = optimize_under_disruption(
        festival_segment_demand,
        bus_disruption=0.0,
        rail_disruption=disruption,
        max_modal_shift=0.10
    )

    rail_disruption_results.append({
        "rail_disruption_pct":
            disruption * 100,

        "bus_services_available":
            summary["bus_services_available"],

        "rail_services_available":
            summary["rail_services_available"],

        "capacity":
            summary["capacity"],

        "passengers_shifted":
            summary["passengers_shifted"],

        "unmet_demand":
            summary["unmet_demand"],

        "coverage_pct":
            summary["coverage_pct"],

        "operating_cost":
            summary["operating_cost"]
    })

rail_disruption_results = pd.DataFrame(
    rail_disruption_results
)

rail_disruption_results

,rail_disruption_pct,bus_services_available,rail_services_available,capacity,passengers_shifted,unmet_demand,coverage_pct,operating_cost
0,0.0,400,100,120000,2395,30864,79.54,1200
1,5.0,400,95,115000,4471,35864,76.23,1160
2,10.0,400,90,110000,4471,40864,72.91,1120
3,15.0,400,85,105000,4471,45864,69.60,1080
4,20.0,400,80,100000,4471,50864,66.28,1040
5,25.0,400,75,95000,3275,55864,62.97,1000
6,30.0,400,70,90000,5279,60864,59.66,960


In [63]:
# ============================================
# 32. COMBINED DISRUPTION SCENARIOS
# ============================================

bus_levels = [0.00, 0.10, 0.20, 0.30]
rail_levels = [0.00, 0.10, 0.20, 0.30]

combined_results = []

for bus_disruption in bus_levels:

    for rail_disruption in rail_levels:

        _, summary = optimize_under_disruption(
            festival_segment_demand,
            bus_disruption=bus_disruption,
            rail_disruption=rail_disruption,
            max_modal_shift=0.10
        )

        combined_results.append({
            "bus_disruption_pct":
                bus_disruption * 100,

            "rail_disruption_pct":
                rail_disruption * 100,

            "bus_services_available":
                summary["bus_services_available"],

            "rail_services_available":
                summary["rail_services_available"],

            "capacity":
                summary["capacity"],

            "unmet_demand":
                summary["unmet_demand"],

            "coverage_pct":
                summary["coverage_pct"]
        })

combined_results = pd.DataFrame(
    combined_results
)

combined_results

,bus_disruption_pct,rail_disruption_pct,bus_services_available,rail_services_available,capacity,unmet_demand,coverage_pct
0,0.0,0.0,400,100,120000,30864,79.54
1,0.0,10.0,400,90,110000,40864,72.91
2,0.0,20.0,400,80,100000,50864,66.28
3,0.0,30.0,400,70,90000,60864,59.66
4,10.0,0.0,360,100,118000,32864,78.22
5,10.0,10.0,360,90,108000,42864,71.59
6,10.0,20.0,360,80,98000,52864,64.96
7,10.0,30.0,360,70,88000,62864,58.33
8,20.0,0.0,320,100,116000,34864,76.89
9,20.0,10.0,320,90,106000,44864,70.26


In [64]:
# ============================================
# 33. RESILIENCE COVERAGE MATRIX
# ============================================

resilience_matrix = combined_results.pivot(
    index="bus_disruption_pct",
    columns="rail_disruption_pct",
    values="coverage_pct"
)

print("RESILIENCE COVERAGE MATRIX")
print("-----------------------------------")

resilience_matrix

RESILIENCE COVERAGE MATRIX
-----------------------------------


rail_disruption_pct,0.0,10.0,20.0,30.0
bus_disruption_pct,,,,
0.0,79.54,72.91,66.28,59.66
10.0,78.22,71.59,64.96,58.33
20.0,76.89,70.26,63.63,57.00
30.0,75.56,68.94,62.31,55.68


## 34. Monte Carlo Resilience Simulation

The deterministic disruption analysis is extended using Monte Carlo
simulation to represent uncertainty in passenger demand and
transport-service availability.

Each simulation samples:

1. Demand variation around the forecast.
2. Bus disruption severity.
3. Rail disruption severity.

The resulting scenario is evaluated through the fleet-constrained
optimization model.

The simulation is used to estimate the probability distribution of
network coverage and unmet demand rather than relying on a single
deterministic scenario.

In [65]:
# ============================================
# 34. MONTE CARLO PARAMETERS
# ============================================

N_SIMULATIONS = 5000

DEMAND_UNCERTAINTY = 0.10

DISRUPTION_LEVELS = [
    0.00,
    0.10,
    0.20,
    0.30
]

DISRUPTION_PROBABILITIES = [
    0.60,
    0.20,
    0.12,
    0.08
]

print("Monte Carlo simulations :", N_SIMULATIONS)
print("Demand uncertainty      :", DEMAND_UNCERTAINTY)
print("Disruption levels       :", DISRUPTION_LEVELS)
print("Disruption probabilities:", DISRUPTION_PROBABILITIES)

Monte Carlo simulations : 5000
Demand uncertainty      : 0.1
Disruption levels       : [0.0, 0.1, 0.2, 0.3]
Disruption probabilities: [0.6, 0.2, 0.12, 0.08]


## 35. Monte Carlo Resilience Simulation

For each simulation:

1. Generate uncertain passenger demand around the forecast.
2. Sample bus disruption severity.
3. Sample rail disruption severity.
4. Re-optimize transportation allocation.
5. Record capacity, unmet demand, and coverage.

The simulation produces a distribution of possible network outcomes.

In [66]:
# ============================================
# 35. MONTE CARLO RESILIENCE SIMULATION
# ============================================

np.random.seed(42)

monte_carlo_results = []

for simulation in range(N_SIMULATIONS):

    # ----------------------------------------
    # 1. Generate uncertain demand
    # ----------------------------------------

    scenario_demand = festival_segment_demand.copy()

    scenario_demand["bus_demand"] = (
        scenario_demand["bus_demand"]
        * np.random.uniform(
            1 - DEMAND_UNCERTAINTY,
            1 + DEMAND_UNCERTAINTY,
            len(scenario_demand)
        )
    )

    scenario_demand["rail_demand"] = (
        scenario_demand["rail_demand"]
        * np.random.uniform(
            1 - DEMAND_UNCERTAINTY,
            1 + DEMAND_UNCERTAINTY,
            len(scenario_demand)
        )
    )

    # ----------------------------------------
    # 2. Sample disruption severity
    # ----------------------------------------

    bus_disruption = np.random.choice(
        DISRUPTION_LEVELS,
        p=DISRUPTION_PROBABILITIES
    )

    rail_disruption = np.random.choice(
        DISRUPTION_LEVELS,
        p=DISRUPTION_PROBABILITIES
    )

    # ----------------------------------------
    # 3. Optimize scenario
    # ----------------------------------------

    _, summary = optimize_under_disruption(
        scenario_demand,
        bus_disruption=bus_disruption,
        rail_disruption=rail_disruption,
        max_modal_shift=0.10
    )

    # ----------------------------------------
    # 4. Store results
    # ----------------------------------------

    monte_carlo_results.append({
        "simulation": simulation + 1,

        "forecast_demand":
            scenario_demand[
                "bus_demand"
            ].sum()
            +
            scenario_demand[
                "rail_demand"
            ].sum(),

        "bus_disruption_pct":
            bus_disruption * 100,

        "rail_disruption_pct":
            rail_disruption * 100,

        "capacity":
            summary["capacity"],

        "passengers_shifted":
            summary["passengers_shifted"],

        "unmet_demand":
            summary["unmet_demand"],

        "coverage_pct":
            summary["coverage_pct"],

        "operating_cost":
            summary["operating_cost"]
    })

monte_carlo_results = pd.DataFrame(
    monte_carlo_results
)

print(
    "Monte Carlo simulations completed:",
    len(monte_carlo_results)
)

display(
    monte_carlo_results.head(10)
)

Monte Carlo simulations completed: 5000


,simulation,forecast_demand,bus_disruption_pct,rail_disruption_pct,capacity,passengers_shifted,unmet_demand,coverage_pct,operating_cost
0,1,149050.840178,10.0,10.0,108000,4526,41051,72.46,1080
1,2,147235.631457,0.0,0.0,120000,5317,28204,80.84,1200
2,3,148786.389736,0.0,0.0,120000,2440,28786,80.65,1200
3,4,151150.806359,10.0,0.0,118000,1807,33151,78.07,1160
4,5,148580.571926,0.0,0.0,120000,3859,28581,80.76,1200
5,6,153803.172177,0.0,0.0,120000,3897,33803,78.02,1200
6,7,149068.451067,0.0,30.0,90000,4352,59068,60.38,960
7,8,151611.136726,0.0,0.0,120000,1350,31611,79.15,1200
8,9,150540.619820,20.0,0.0,116000,2791,34541,77.06,1120
9,10,152548.268601,0.0,0.0,120000,1106,32548,78.66,1200


In [67]:
# ============================================
# 36. MONTE CARLO SUMMARY STATISTICS
# ============================================

mc_summary = pd.DataFrame({
    "Metric": [
        "Forecast demand",
        "Capacity",
        "Unmet demand",
        "Coverage",
        "Operating cost"
    ],

    "Mean": [
        monte_carlo_results["forecast_demand"].mean(),
        monte_carlo_results["capacity"].mean(),
        monte_carlo_results["unmet_demand"].mean(),
        monte_carlo_results["coverage_pct"].mean(),
        monte_carlo_results["operating_cost"].mean()
    ],

    "Std Dev": [
        monte_carlo_results["forecast_demand"].std(),
        monte_carlo_results["capacity"].std(),
        monte_carlo_results["unmet_demand"].std(),
        monte_carlo_results["coverage_pct"].std(),
        monte_carlo_results["operating_cost"].std()
    ],

    "Minimum": [
        monte_carlo_results["forecast_demand"].min(),
        monte_carlo_results["capacity"].min(),
        monte_carlo_results["unmet_demand"].min(),
        monte_carlo_results["coverage_pct"].min(),
        monte_carlo_results["operating_cost"].min()
    ],

    "Maximum": [
        monte_carlo_results["forecast_demand"].max(),
        monte_carlo_results["capacity"].max(),
        monte_carlo_results["unmet_demand"].max(),
        monte_carlo_results["coverage_pct"].max(),
        monte_carlo_results["operating_cost"].max()
    ]
})

mc_summary

,Metric,Mean,Std Dev,Minimum,Maximum
0,Forecast demand,150835.531907,3197.690682,140557.408359,161493.382236
1,Capacity,111900.600000,9826.496407,84000.000000,120000.000000
2,Unmet demand,39050.549800,10266.996905,24077.000000,72761.000000
3,Coverage,74.143694,6.650630,53.590000,83.160000
4,Operating cost,1118.884800,86.096776,840.000000,1200.000000


In [68]:
# ============================================
# 37. MONTE CARLO RISK INDICATORS
# ============================================

coverage = monte_carlo_results["coverage_pct"]
unmet = monte_carlo_results["unmet_demand"]

risk_indicators = {
    "Mean coverage (%)":
        coverage.mean(),

    "Median coverage (%)":
        coverage.median(),

    "5th percentile coverage (%)":
        coverage.quantile(0.05),

    "10th percentile coverage (%)":
        coverage.quantile(0.10),

    "95th percentile unmet demand":
        unmet.quantile(0.95),

    "Probability coverage < 70%":
        (coverage < 70).mean() * 100,

    "Probability coverage < 75%":
        (coverage < 75).mean() * 100,

    "Probability unmet demand > 30000":
        (unmet > 30000).mean() * 100,

    "Worst simulated coverage (%)":
        coverage.min(),

    "Worst simulated unmet demand":
        unmet.max()
}

print("MONTE CARLO RISK INDICATORS")
print("-----------------------------------")

for key, value in risk_indicators.items():

    print(
        f"{key}: {value:.2f}"
    )

MONTE CARLO RISK INDICATORS
-----------------------------------
Mean coverage (%): 74.14
Median coverage (%): 76.68
5th percentile coverage (%): 59.42
10th percentile coverage (%): 63.59
95th percentile unmet demand: 61257.75
Probability coverage < 70%: 22.72
Probability coverage < 75%: 41.24
Probability unmet demand > 30000: 83.42
Worst simulated coverage (%): 53.59
Worst simulated unmet demand: 72761.00
